# 第9章「データベースとデータ分析の橋渡し：前処理のためのpandas入門」

以下，9章に登場するコードを掲載します．
なお実行環境として，以下を想定して動作確認を行なっています（カッコ内はバージョンです）．

- Python (>= `3.13.11`)
- SQLAlchemy (>= `2.0.49`)
- sciki-tlearn (>= `1.8.0`)
- pandas (>= `2.2.3`)

### データのダウンロード

In [1]:
!wget https://github.com/trycycle/database-textbook/raw/main/dataset/sql2-example.db

--2026-11-20 13:59:05--  https://github.com/trycycle/database-textbook/raw/main/dataset/sql2-example.db
github.com (github.com) をDNSに問いあわせています... 20.27.177.113
github.com (github.com)|20.27.177.113|:443 に接続しています... 接続しました。
HTTP による接続要求を送信しました、応答を待っています... 302 Found
場所: https://raw.githubusercontent.com/trycycle/database-textbook/main/dataset/sql2-example.db [続く]
--2026-11-20 13:59:06--  https://raw.githubusercontent.com/trycycle/database-textbook/main/dataset/sql2-example.db
raw.githubusercontent.com (raw.githubusercontent.com) をDNSに問いあわせています... 185.199.108.133, 185.199.111.133, 185.199.109.133, ...
raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443 に接続しています... 接続しました。
HTTP による接続要求を送信しました、応答を待っています... 200 OK
長さ: 327680 (320K) [application/octet-stream]
`sql2-example.db' に保存中

sql2-example.db     100%[===================>] 320.00K  --.-KB/s 時間 0.01s      

2026-11-20 13:59:07 (27.6 MB/s) - `sql2-example.db' へ保存完了 [327680/327680]



### コード9.1：ライブラリのインストール
ターミナル上でPythonスクリプトを実行する場合は，以下のコマンドを使用して必要なライブラリをインストールしてください

```
pip install pandas sqlalchemy scikit-learn
```

Jupyter Notebook上でPythonスクリプトを実行する場合は，Notebookのセルで以下のコマンドを実行してください．

```
!pip install pandas sqlalchemy scikit-learn
```


### コード9.2：ライブラリのインポートと関係データベースへの接続

In [2]:
import pandas as pd
import sqlalchemy

# SQLiteデータベースへの接続エンジンを作成する
db_path = 'sql2-example.db'
engine = sqlalchemy.create_engine(f'sqlite:///{db_path}')

### コード9.3：read_sql()によるデータベースからの読み込み

In [3]:
# 「商品」テーブルの全レコードを取得する
sql = 'SELECT * FROM 商品;'
df_products = pd.read_sql(sql, engine)

df_products

,商品ID,商品名,カテゴリ,メーカー,価格,在庫数
0,P001,国産カレーパン,パン,A製パン,270,206
1,P002,職人仕込みクロワッサン,パン,Bフーズ,230,443
2,P003,国産チキンカレー,レトルト,C食品,420,149
3,P004,天然中華丼の素,レトルト,Dフード,180,166
4,P005,味わい焼肉のたれ,調味料,E食品,890,18
...,...,...,...,...,...,...
495,P496,プレミアムオレンジジュース,飲料,Bフーズ,300,468
496,P497,有機カレーパン,パン,A製パン,520,367
497,P498,贅沢ウーロン茶,飲料,Bフーズ,300,52
498,P499,味わいビスケット,菓子,Bフーズ,250,203


### コード9.4：データフレームの概要を確認するPythonコード

In [4]:
# データフーレムの行数と列数を返す
df_products.shape

(500, 6)

In [5]:
# 列名の一覧を返す
df_products.columns

Index(['商品ID', '商品名', 'カテゴリ', 'メーカー', '価格', '在庫数'], dtype='object')

In [6]:
# 列毎のデータ型と非NULL値の数を表示する
df_products.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   商品ID    500 non-null    object
 1   商品名     500 non-null    object
 2   カテゴリ    500 non-null    object
 3   メーカー    500 non-null    object
 4   価格      500 non-null    int64 
 5   在庫数     500 non-null    int64 
dtypes: int64(2), object(4)
memory usage: 23.6+ KB


In [7]:
# 数値列の基本統計量を表示する
df_products.describe()

,価格,在庫数
count,500.000000,500.000000
mean,390.660000,255.194000
std,234.081509,142.358027
min,100.000000,2.000000
25%,210.000000,130.750000
50%,330.000000,261.500000
75%,510.000000,368.500000
max,1340.000000,500.000000


### コード9.5：列を抽出するPythonコード


In [8]:
# 1 つの列だけを取り出す
names = df_products['商品名']

# 複数の列を取り出す
subset_df = df_products[['商品名','価格','カテゴリ']]

# 先頭5件を表示する
subset_df.head()

,商品名,価格,カテゴリ
0,国産カレーパン,270,パン
1,職人仕込みクロワッサン,230,パン
2,国産チキンカレー,420,レトルト
3,天然中華丼の素,180,レトルト
4,味わい焼肉のたれ,890,調味料


### コード9.6：行を絞り込むPythonコード

In [9]:
# (1) 価格が500以上の商品のみを取り出す
expensive_items = df_products[df_products['価格'] >= 500]

# 先頭5件を表示する
expensive_items.head()

,商品ID,商品名,カテゴリ,メーカー,価格,在庫数
4,P005,味わい焼肉のたれ,調味料,E食品,890,18
10,P011,無添加麻婆豆腐の素,レトルト,Fファーム,520,299
14,P015,濃厚ごまドレッシング,調味料,Fファーム,650,57
18,P019,贅沢焼肉のたれ,調味料,E食品,890,391
20,P021,味わい中華丼の素,レトルト,C食品,550,254


In [10]:
# (2) 文字列の条件検索：商品名に「カレー」を含む商品のみを取り出す
curry_items = df_products[df_products['商品名'].str.contains('カレー')]

# 先頭5件を表示する
curry_items.head()

,商品ID,商品名,カテゴリ,メーカー,価格,在庫数
0,P001,国産カレーパン,パン,A製パン,270,206
2,P003,国産チキンカレー,レトルト,C食品,420,149
36,P037,じっくり煮込んだカレーパン,パン,A製パン,270,13
67,P068,濃厚ビーフカレー,レトルト,L食品,800,422
68,P069,濃厚チキンカレー,レトルト,C食品,230,268


In [11]:
# (3) queryメソッドを使うとSQLライクに書けて読みやすい
expensive_seasonings = df_products.query('価格 >= 500 & カテゴリ == "調味料"')

expensive_seasonings.head()

,商品ID,商品名,カテゴリ,メーカー,価格,在庫数
4,P005,味わい焼肉のたれ,調味料,E食品,890,18
14,P015,濃厚ごまドレッシング,調味料,Fファーム,650,57
18,P019,贅沢焼肉のたれ,調味料,E食品,890,391
31,P032,辛口醤油,調味料,L食品,530,440
43,P044,辛口味噌,調味料,E食品,510,403


### コード9.7： 列の追加と計算を行うPythonコード

In [12]:
# 「価格」と「在庫数」の列をかけ算して新しい列「在庫金額」を作る
df_products['在庫金額'] = df_products['価格'] * df_products['在庫数']

# 先頭5件を表示する
df_products.head()

,商品ID,商品名,カテゴリ,メーカー,価格,在庫数,在庫金額
0,P001,国産カレーパン,パン,A製パン,270,206,55620
1,P002,職人仕込みクロワッサン,パン,Bフーズ,230,443,101890
2,P003,国産チキンカレー,レトルト,C食品,420,149,62580
3,P004,天然中華丼の素,レトルト,Dフード,180,166,29880
4,P005,味わい焼肉のたれ,調味料,E食品,890,18,16020


### コード9.8：applyによる関数を適用するPythonコード

In [13]:
# 価格に応じて「高価格」「中価格」「低価格」の3つに分類する関数を定義する
def classify_price(p):
    if p >= 500:
        return '高価格'
    elif p >= 300:
        return '中価格'
    else:
        return '低価格'


# apply メソッドで関数を列全体に適用する
df_products['価格帯'] = df_products['価格'].apply(classify_price)

# ラムダ式（無名関数）を使って1 行で簡潔に書くこともできる
# 例：500 円以上の商品だけを1 割引きにする
df_products['割引後価格'] = df_products['価格'].apply(
    lambda p: p * 0.9 if p >= 500 else p
)

# 先頭5件を表示する
df_products.head()

,商品ID,商品名,カテゴリ,メーカー,価格,在庫数,在庫金額,価格帯,割引後価格
0,P001,国産カレーパン,パン,A製パン,270,206,55620,低価格,270.0
1,P002,職人仕込みクロワッサン,パン,Bフーズ,230,443,101890,低価格,230.0
2,P003,国産チキンカレー,レトルト,C食品,420,149,62580,中価格,420.0
3,P004,天然中華丼の素,レトルト,Dフード,180,166,29880,低価格,180.0
4,P005,味わい焼肉のたれ,調味料,E食品,890,18,16020,高価格,801.0


### コード9.9：groupbyによる集計を行うPythonコード

In [14]:
# カテゴリごとの在庫数の合計を求める
stock_sum = df_products.groupby('カテゴリ')[['在庫数']].sum()

# 先頭5件を表示する
stock_sum.head()

,在庫数
カテゴリ,
パン,21281
レトルト,25113
菓子,28248
調味料,27997
飲料,24958


In [15]:
# カテゴリごとの価格の平均と最大値を一度に求める
price_stats = df_products.groupby('カテゴリ').agg({'価格': ['mean', 'max']})

# 先頭5件を表示する
price_stats.head()

価格      
            mean   max
カテゴリ                  
パン    312.588235   590
レトルト  501.078431  1040
菓子    278.584906   570
調味料   660.392157  1340
飲料    197.714286   360

### コード9.10：value_countsによる頻度集計を行うPythonコード

In [16]:
# カテゴリごとの商品数を求める
df_products['カテゴリ'].value_counts()

カテゴリ
菓子      106
飲料      105
レトルト    102
調味料     102
パン       85
Name: count, dtype: int64

### コード9.11：mergeによる横方向の結合を行うPythonコード

In [17]:
# 注文テーブルを読み込む
sql = 'SELECT * FROM 注文;'
df_orders = pd.read_sql(sql, engine)

# 商品テーブルを（再度）読み込む
sql = 'SELECT * FROM 商品;'
df_products = pd.read_sql(sql, engine)

# 「商品ID」をキーにして、商品テーブルと注文テーブルを結合する
df_merged = pd.merge(df_orders, df_products, on='商品ID', how='inner')

# 先頭5件を表示する
df_merged.head()

,注文ID,ユーザID,商品ID,数量,注文日,商品名,カテゴリ,メーカー,価格,在庫数
0,O0001,U006,P182,1,2024-02-10,じっくり煮込んだトマトジュース,飲料,Fファーム,200,452
1,O0002,U016,P276,2,2022-10-20,昔ながらのフルーツグミ,菓子,N製菓,310,117
2,O0003,U057,P178,5,2025-01-03,カロリーオフ塩クラッカー,菓子,N製菓,300,125
3,O0004,U021,P361,10,2023-10-29,ふんわり麦茶,飲料,Bフーズ,220,327
4,O0005,U069,P311,7,2024-12-26,濃厚フルーツグミ,菓子,K製菓,110,447


### コード9.12：concatによる縦方向の結合を行うPythonコード

In [18]:
# 2024年より前の注文情報が収められたテーブル「注文_2024以前」を読み込む
df_orders_before_2024 = pd.read_sql('SELECT * FROM 注文_2024以前;', engine)

# 2024年以降の注文情報が収められたテーブル「注文_2024以後」を読み込む
df_orders_after_2024 = pd.read_sql('SELECT * FROM 注文_2024以後;', engine)

# 2024年以前の注文データと2024年以降の注文データを縦につなぐ
# ignore_index=True を指定してインデックスを振り直す
df_orders = pd.concat(
    [df_orders_before_2024, df_orders_after_2024], ignore_index=True)

### コード9.13：欠損値を処理し，df_mergedの一部を穴埋めするPythonコード

In [19]:
sql = """
SELECT
    *
FROM
    商品
LEFT OUTER JOIN 注文
USING(商品ID);
"""

# 「商品」テーブルと「注文」テーブルを「商品ID」で外部結合したものをデータフレームに
df_merged = pd.read_sql(sql, engine)

#(1) 数量がNaN（注文されたことがない商品）のレコードを表示
df_merged[df_merged['数量'].isna()]

,商品ID,商品名,カテゴリ,メーカー,価格,在庫数,注文ID,ユーザID,数量,注文日
584,P142,こだわり和風ドレッシング,調味料,Fファーム,880,187,None,None,NaN,None
1441,P365,無添加のど飴,菓子,I製菓,370,284,None,None,NaN,None


In [20]:
# NaNが含まれる行をまるごと削除する（情報が失われるので注意）
df_clean = df_merged.dropna()

# NaNを特定の値（例えば0）で穴埋めする
df_merged['数量'] = df_merged['数量'].fillna(0)

#(2) 数量がNaNであったレコードを欠損値を埋めた後はどうなったか？
df_merged[(df_merged.商品ID == 'P142') | (df_merged.商品ID == 'P365')]

,商品ID,商品名,カテゴリ,メーカー,価格,在庫数,注文ID,ユーザID,数量,注文日
584,P142,こだわり和風ドレッシング,調味料,Fファーム,880,187,None,None,0.0,None
1441,P365,無添加のど飴,菓子,I製菓,370,284,None,None,0.0,None


### コード9.14：ダミー変数化のためのPythonコード

In [21]:
# 商品テーブルを（再度）読み込む
sql = 'SELECT * FROM 商品;'
df_products = pd.read_sql(sql, engine)

# 「カテゴリ」列をダミー変数化する
df_encoded = pd.get_dummies(df_products, columns=['カテゴリ'], dtype='int')

# 先頭5件を表示する
df_encoded.head()

,商品ID,商品名,メーカー,価格,在庫数,カテゴリ_パン,カテゴリ_レトルト,カテゴリ_菓子,カテゴリ_調味料,カテゴリ_飲料
0,P001,国産カレーパン,A製パン,270,206,1,0,0,0,0
1,P002,職人仕込みクロワッサン,Bフーズ,230,443,1,0,0,0,0
2,P003,国産チキンカレー,C食品,420,149,0,1,0,0,0
3,P004,天然中華丼の素,Dフード,180,166,0,1,0,0,0
4,P005,味わい焼肉のたれ,E食品,890,18,0,0,0,1,0


### コード9.15：scikit-learnを用いたスケーリングを行うPythonコード

In [22]:
# 商品テーブルを（再度）読み込む
sql = 'SELECT * FROM 商品;'
df_products = pd.read_sql(sql, engine)

In [23]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler

# 正規化（MinMaxScaler）
# 価格と在庫数を正規化し，新しい列を追加
scaler_minmax = MinMaxScaler()
df_products[['価格_正規化', '在庫数_正規化']] = scaler_minmax.fit_transform(
        df_products[['価格', '在庫数']])

# 正規化（StandardScaler）
# 価格と在庫数を標準化し，新しい列を追加
scaler_std = StandardScaler()
df_products[['価格_標準化', '在庫数_標準化']] = scaler_std.fit_transform(
        df_products[['価格', '在庫数']])

# 先頭5件を表示する
df_products.head()

,商品ID,商品名,カテゴリ,メーカー,価格,在庫数,価格_正規化,在庫数_正規化,価格_標準化,在庫数_標準化
0,P001,国産カレーパン,パン,A製パン,270,206,0.137097,0.409639,-0.515978,-0.345911
1,P002,職人仕込みクロワッサン,パン,Bフーズ,230,443,0.104839,0.885542,-0.687029,1.320572
2,P003,国産チキンカレー,レトルト,C食品,420,149,0.258065,0.295181,0.125466,-0.746711
3,P004,天然中華丼の素,レトルト,Dフード,180,166,0.064516,0.329317,-0.900844,-0.627175
4,P005,味わい焼肉のたれ,調味料,E食品,890,18,0.637097,0.032129,2.135325,-1.667848
